# Chapter 2: Data Ingestion and Preparation

## 1. Introduction

Welcome to the cornerstone of any data analysis workflow: getting data into your environment. In precision health, critical information—from patient vitals and lab results to medication logs—is often stored in simple, universal file formats like CSV and Excel. This section establishes the foundational skill of loading this data into Pandas DataFrames. We will begin by reading standard, clean files and progressively build up to handling the complex, messy data structures commonly encountered in real-world clinical and research settings, ensuring your data is accurate and ready for analysis from the very first step.

---

## 2. Key Concepts and Definitions

*   **DataFrame**: The primary data structure in Pandas, analogous to a digital spreadsheet or a patient's medical chart. Each row represents an observation (e.g., a single clinic visit), and each column represents a variable (e.g., heart rate, temperature).
*   **CSV (Comma-Separated Values)**: A plain text file format used to store tabular data. Each line is a data record, and each record consists of one or more fields, separated by commas. In a medical context, a daily export of patient vitals from a monitoring device is often in CSV format.
*   **Excel File (.xlsx)**: A spreadsheet file format from Microsoft Excel that can contain multiple worksheets, formatting, and formulas. A hospital pharmacy might use an Excel file with different sheets to track monthly medication inventories for different wards.
*   **Delimiter**: The character used to separate columns in a flat file. While a comma is standard for CSVs, other delimiters like tabs (`\t`) or semicolons (`;`) are common in outputs from lab instruments. Specifying the correct delimiter is crucial for parsing the file correctly.
*   **Header**: The first row in a data file that contains the names of the columns. A file without a header requires you to assign column names manually during import to give the data a meaningful structure.
*   **Schema on Read**: The practice of defining the structure and data types of columns as you load a file. This is a proactive data integrity technique, like ensuring a `patient_id` with leading zeros (e.g., 'PT000123') is always treated as text (`string`) and not an integer, preventing patient misidentification.

---

## 3. Main Content

### 3.1 Reading a Standard CSV File

The most common data import task is loading a clean, comma-separated file. We use the `pd.read_csv()` function. This is the first step in analyzing data like daily patient vitals exported from an electronic health record (EHR) system.

In [ ]:
import pandas as pd
import io

# Simulate a CSV file in memory to make the example runnable
csv_data = """patient_id,age,heart_rate_bpm,temperature_celsius
PT000001,68,72,37.1
PT000002,54,81,38.2
PT000003,76,68,36.8
PT000004,34,88,37.5
PT000005,81,70,36.9
"""

vitals_df = pd.read_csv(io.StringIO(csv_data))
print(vitals_df.head())

# Expected Output:
#   patient_id  age  heart_rate_bpm  temperature_celsius
# 0   PT000001   68              72                 37.1
# 1   PT000002   54              81                 38.2
# 2   PT000003   76              68                 36.8
# 3   PT000004   34              88                 37.5
# 4   PT000005   81              70                 36.9

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




*   `pd.read_csv()`: This is the primary function for loading data from a CSV file into a Pandas DataFrame.
*   `.head()`: A quick and essential method to display the first five rows, allowing you to visually inspect the DataFrame and confirm the import was successful.

While CSVs are common, much of the administrative and financial data in healthcare is stored in Excel files. Let's see how to access a specific worksheet within an Excel workbook.

### 3.2 Reading a Specific Excel Sheet

Clinical data is often organized across multiple worksheets in a single Excel file (e.g., weekly pharmacy logs, quarterly trial results). `pd.read_excel()` allows you to target a specific sheet by name or index.

> **Important:** Reading modern Excel files (`.xlsx`) requires the `openpyxl` library. Before running this code, make sure you have installed it from your terminal with `pip install openpyxl`. This is a one-time setup for your environment.

In [ ]:
import pandas as pd

# This example requires a local file named 'medication_log.xlsx'
# with a sheet named 'Week2'.
medication_df = pd.read_excel(
    'medication_log.xlsx',
    sheet_name='Week2',
    engine='openpyxl'
)
print(medication_df.head())

# Expected Output:
# (Assuming 'medication_log.xlsx' has columns like 'drug_id', 'patient_id', 'dosage')
#    drug_id  patient_id  dosage
# 0   DRG-A5      PT00451     10mg
# 1   DRG-B2      PT00239     25mg
# ...

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




*   `sheet_name='Week2'`: Specifies which worksheet to load. You can use the sheet's name or its numerical index (e.g., `sheet_name=1` for the second sheet).
*   `engine='openpyxl'`: This parameter explicitly tells Pandas to use the `openpyxl` library, which is the modern standard for reading and writing `.xlsx` files.

Real-world data, especially from lab instruments, is rarely as clean as the examples above. Next, we'll learn to handle files with extra metadata and non-standard formatting.

### 3.3 Handling Complex File Structures

> **In Practice:** Raw data from laboratory instruments or legacy systems often includes metadata headers with information like the device ID, calibration date, or batch number. Learning to programmatically skip these rows is a crucial skill for automating data intake and avoiding manual file cleaning.

`read_csv()` has powerful parameters to handle files with metadata at the top or that use non-standard delimiters.

In [ ]:
import pandas as pd
import io

# Simulate a tab-separated file (.tsv) with metadata headers
tsv_data = """Report Generated: 2025-11-12
Device ID: LAB-04
---
PT000101\tglucose\t105\tmg/dL\tTechA
PT000102\tHemoglobin A1c (HbA1c)\tNA\t%\tTechB
"""

lab_df_structure = pd.read_csv(
    io.StringIO(tsv_data),
    sep='\t',
    skiprows=3
)
print(lab_df_structure.head())

# Expected Output:
#   PT000101  glucose  105  mg/dL  TechA
# 0  PT000102  Hemoglobin A1c (HbA1c)  NaN      %  TechB

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




*   `sep='\t'`: Specifies that the columns are separated by a tab character (`\t`) instead of a comma.
*   `skiprows=3`: Instructs Pandas to ignore the first three lines of the file and start reading data from the fourth line.

> **Debug Note:** If you get a `ParserError`, it often means the `sep` (separator) you specified doesn't match what's in the file. A common mistake is using the default comma separator for a file that is actually tab-separated (`\t`) or semicolon-separated (`;`).

We've successfully ignored the metadata, but notice the output: Pandas has incorrectly assigned the first data row as the header. Next, we will define a proper schema during import to correctly structure our data.

### 3.4 Defining Data Schema on Import

To ensure data integrity, especially when a file lacks a header, you can define the column names, data types, and missing value representations during the import process.

In [ ]:
import pandas as pd
import io

# Continuing with the same tsv_data
tsv_data = """Report Generated: 2025-11-12
Device ID: LAB-04
---
PT000101\tglucose\t105\tmg/dL\tTechA
PT000102\tHemoglobin A1c (HbA1c)\tNA\t%\tTechB
"""

lab_df_schema = pd.read_csv(
    io.StringIO(tsv_data),
    sep='\t',
    skiprows=3,
    header=None,
    names=['patient_id', 'test', 'value', 'unit', 'tech_id'],
    na_values=['NA'],
    dtype={'patient_id': str}
)
print(lab_df_schema.head())

# Expected Output:
#   patient_id                      test  value   unit tech_id
# 0   PT000101                   glucose  105.0  mg/dL   TechA
# 1   PT000102  Hemoglobin A1c (HbA1c)    NaN      %   TechB

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




*   `header=None`: Informs Pandas that the file has no header row.
*   `names=[...]`: Provides a list of strings to use as column names.
*   `na_values=['NA']`: Specifies that the string 'NA' should be interpreted as a missing value (`NaN`).
*   `dtype={'patient_id': str}`: Forces the `patient_id` column to be read as a string.

> **Important:** Forcing `patient_id` to a string with `dtype={'patient_id': str}` is critical for patient safety and data integrity. Many medical record numbers (MRNs) use leading zeros (e.g., '001234'). If read as a number, Python would interpret this as `1234`, potentially leading to patient misidentification when merging datasets.

Now that our data is correctly structured, let's see how to make the loading process more efficient, which is vital for large clinical datasets.

### 3.5 Optimizing Data Loading

> **Pro Tip:** When working with massive clinical datasets, such as genomics or large-scale electronic health records (EHR) exports, loading only the necessary columns with `usecols` can dramatically reduce memory consumption and speed up your analysis.

You can optimize loading by selecting only the columns you need and setting an index column directly.

In [ ]:
import pandas as pd
import io

# Continuing with the same tsv_data
tsv_data = """Report Generated: 2025-11-12
Device ID: LAB-04
---
PT000101\tglucose\t105\tmg/dL\tTechA
PT000102\tHemoglobin A1c (HbA1c)\tNA\t%\tTechB
"""

lab_df_final = pd.read_csv(
    io.StringIO(tsv_data),
    sep='\t',
    skiprows=3,
    header=None,
    names=['patient_id', 'test', 'value', 'unit', 'tech_id'],
    na_values=['NA'],
    dtype={'patient_id': str},
    usecols=['patient_id', 'test', 'value', 'unit'] # Load only these columns
)

# Set the index after loading (preferred over inplace=True)
lab_df_final = lab_df_final.set_index('patient_id')
print(lab_df_final.head())

# Expected Output:
#                                      test  value   unit
# patient_id
# PT000101                         glucose  105.0  mg/dL
# PT000102        Hemoglobin A1c (HbA1c)    NaN      %

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




*   `usecols=[...]`: A memory-saving parameter that instructs Pandas to load only a specific subset of columns.
*   `lab_df_final = lab_df_final.set_index('patient_id')`: Sets a column as the DataFrame's index. Reassigning the variable is the modern, preferred practice over using `inplace=True`.

> **Pro Tip:** You can set the index directly during import by using the `index_col` parameter (e.g., `index_col='patient_id'`). This is slightly more efficient than using `set_index()` after loading. We teach `set_index()` separately as it's a versatile function you'll use in many other contexts.

---

## 4. Practice Exercises

### Exercise 1: Basic CSV Import

**Objective:** Perform a standard import of a CSV file with clinically realistic columns.
**Time:** 3 minutes
**Medical Context:** As part of a daily quality check at City Medical Center, you need to perform a quick inspection of the latest patient vitals.

Load the file `daily_vitals.csv`. The columns are `patient_id`, `timestamp` (in 'YYYY-MM-DD HH:MM:SS' format), `temperature_celsius`, `bp_systolic`, and `bp_diastolic`. Name the DataFrame `vitals_df` and display the first five rows.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
import pandas as pd

# This solution assumes a file 'daily_vitals.csv' exists.
# Example content:
# patient_id,timestamp,temperature_celsius,bp_systolic,bp_diastolic
# PT000201,2025-11-13 08:00:00,36.8,120,80

# Load the CSV file into a DataFrame
vitals_df = pd.read_csv('daily_vitals.csv')

# Display the first five rows to verify
print(vitals_df.head())
```
**Explanation:** This uses `pd.read_csv()` with just the file path, as the file is a standard, well-formatted CSV with a header row.
**Key Learning:** Basic file loading with `pd.read_csv()` is simple and effective for clean, structured clinical data.


</div>
</details>

### Exercise 2: Reading a Specific Excel Sheet

**Objective:** Load data from a specific worksheet within an Excel file.
**Time:** 5 minutes
**Medical Context:** The pharmacy has sent over its quarterly order logs. Your manager, Dr. Chen, has asked you to analyze only the second quarter's data.

An Excel file named `pharmacy_orders.xlsx` contains two sheets: 'Orders_Q1' and 'Orders_Q2'. Load only the data from the 'Orders_Q2' sheet into a DataFrame called `q2_orders_df`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
import pandas as pd

# This solution requires 'pip install openpyxl'
# and a local file 'pharmacy_orders.xlsx'.

# Load the 'Orders_Q2' sheet from the Excel file
q2_orders_df = pd.read_excel(
    'pharmacy_orders.xlsx',
    sheet_name='Orders_Q2',
    engine='openpyxl'
)

# Display the head of the new DataFrame
print(q2_orders_df.head())
```
**Explanation:** The `sheet_name` parameter is used to specify which worksheet to import from the Excel workbook.
**Key Learning:** Use `pd.read_excel()` with the `sheet_name` argument to target specific data within a multi-sheet workbook.


</div>
</details>

### Exercise 3: Advanced CSV Import with Schema Definition

**Objective:** Import a messy, tab-separated file while defining the schema and setting an index.
**Time:** 7 minutes
**Medical Context:** A collaborating lab has sent results in a poorly formatted `lab_results.tsv` file. To prepare it for merging with your main patient database, you must load the file while assigning correct column names, handling missing values, and setting the `patient_id` as the index for efficient lookups.

Load the tab-separated file `lab_results.tsv`. The file has no header and uses `'MISSING'` for nulls. The columns should be `['patient_id', 'test_type', 'result']`. Set `patient_id` as the index.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
import pandas as pd
import io

# Simulate the messy file
messy_data = """PT000333\tWBC\t8.1
PT000444\tRBC\tMISSING
PT000555\tPlatelet\t250
"""

# Define column names
col_names = ['patient_id', 'test_type', 'result']

# Load the data using advanced parameters and set index in one step
lab_df = pd.read_csv(
    io.StringIO(messy_data),
    sep='\t',
    header=None,
    names=col_names,
    na_values=['MISSING'],
    index_col='patient_id'
)

# Display the head of the final DataFrame
print(lab_df.head())

# Expected Output:
#            test_type  result
# patient_id
# PT000333         WBC     8.1
# PT000444         RBC     NaN
# PT000555    Platelet   250.0
```
**Explanation:** We combine multiple parameters: `sep='\t'` for the delimiter, `header=None` because there's no header row, `names` to assign our own column titles, `na_values` to correctly handle missing data, and `index_col` to set the index upon loading.
**Key Learning:** Chaining parameters in `pd.read_csv()` provides powerful control for importing complex and poorly structured files cleanly.


</div>
</details>

> **Reflection Moment:** Think about the data you work with regularly. Does it come in clean, structured files, or does it resemble the "messy" data from our lab report example? How could the parameters we learned today (`skiprows`, `sep`, `na_values`) help automate your own data-loading workflows?

---

## 5. Practical Applications

*   **Aggregating Clinical Trial Data:** Researchers often receive patient data from multiple trial sites as separate CSV files. Using a loop and `pd.read_csv()`, they can ingest all files into a single master DataFrame, standardizing column names with the `names` parameter to ensure consistency for a meta-analysis.
*   **Analyzing Pharmacy Dispensation Logs:** A hospital analyst can use `pd.read_excel()` with the `sheet_name` parameter to pull monthly medication dispensation data from a yearly workbook. This allows them to track drug usage, identify trends in prescriptions, and manage inventory without manually separating the data.
*   **Processing Genomic Sequencer Output:** Genomic data files are often massive, tab-delimited, and prefixed with many lines of metadata. A bioinformatician would use `pd.read_csv()` with `sep='\t'`, `skiprows`, and `usecols` to efficiently parse these files, ignoring the metadata and loading only the essential gene and expression columns to conserve memory.
*   **Integrating EHR Data Exports:** When a hospital exports a large dataset from its Electronic Health Record (EHR) system, patient IDs (e.g., '00123') might be incorrectly converted to numbers (123). Using `dtype={'patient_id': str}` during import prevents this data corruption and ensures patient records can be accurately joined later.

---

## 6. Summary and Key Takeaways

In this section, we've explored the essential functions for loading data from the most common file formats in healthcare analytics. We learned how to move beyond simple file reads to skillfully handle the complexities of real-world data, ensuring a solid foundation for any analysis.

*   **Use `pd.read_csv()` for CSV files and `pd.read_excel()` for Excel files.** These are your

---